# 🛰️ Detección de Cambios en el Terreno — Área Metropolitana de Bucaramanga

**Detección de cambios en terrenos mediante el análisis de imágenes satelitales multitemporales**

Este notebook implementa, **fase a fase**, una herramienta para detectar cambios en la cobertura
terrestre (expansión urbana, pérdida/ganancia de vegetación, variaciones en cuerpos de agua) en el
**Área Metropolitana de Bucaramanga (AMB)** — Bucaramanga, Floridablanca, Girón y Piedecuesta —,
a partir de imágenes satelitales **Sentinel-2** de acceso abierto, procesadas con **Google Earth Engine (GEE)**.

## Objetivo general
Desarrollar una herramienta para la detección de cambios en la cobertura terrestre mediante el
análisis de imágenes satelitales multitemporales, aplicando técnicas de procesamiento digital de
imágenes, con la finalidad de mejorar la eficiencia del monitoreo ambiental en un área específica.

## Estructura del notebook (una fase por cada objetivo específico)

| Fase | Objetivo específico | Contenido |
|---|---|---|
| **Fase 0** | — | Configuración del entorno (librerías y conexión a Google Earth Engine) |
| **Fase 1** | Recopilar imágenes satelitales multitemporales | Definición del área de estudio (AMB) y búsqueda de imágenes Sentinel-2 en varios periodos |
| **Fase 2** | Preprocesar las imágenes | Enmascarado de nubes, composición sin nubes e índices espectrales (NDVI, NDBI, NDWI) |
| **Fase 3** | Sistematizar la detección de cambios | Diferencias de índices, umbral estadístico reproducible, mapa de cambios clasificado **y comparaciones año a año** |
| **Fase 4** | Evaluar y validar los resultados | Estadísticas de área (principal y año a año), validación multiclase con datos independientes (matriz de confusión, OA, AA y Kappa) y análisis de sensibilidad |
| **Fase 5** | Generar visualizaciones | Mapa interactivo, comparaciones antes/después, **panel de mapas de cambio año a año**, series de tiempo y exportación de resultados |

## Antes de empezar

1. **No necesitas GPU.** Todo el procesamiento pesado ocurre en los servidores de Google Earth Engine; Colab solo orquesta las solicitudes.
2. Necesitas una **cuenta de Google Earth Engine** (gratuita para uso no comercial/académico):
   - Regístrate en https://code.earthengine.google.com/register si no lo has hecho antes.
   - Crea (o reutiliza) un **proyecto de Google Cloud** asociado a Earth Engine; necesitarás su *ID de proyecto* en la Fase 0.
3. Ejecuta las celdas **en orden, de arriba hacia abajo** (▶️). Cada fase depende de las variables creadas en las fases anteriores.
4. El área de estudio y los periodos temporales son **parámetros configurables** (Fase 1): puedes ajustarlos a otra región o a otros años sin modificar el resto del notebook.
5. Al final (Fase 5) se generan y descargan: un mapa de cambios (GeoTIFF), figuras (PNG) y una tabla de estadísticas (CSV).
6. La Fase 1.6 guarda automáticamente el conjunto de imágenes recopiladas (una por periodo) en tu
   Google Drive, en `MyDrive/deteccion_cambios_bucaramanga/01_imagenes_recopiladas/` — esa carpeta
   ya existe y coincide con `CARPETA_RESULTADOS` (Fase 0.3), así que no necesitas crearla a mano.
7. El notebook trabaja con **8 periodos anuales (2017-2024)** por defecto, no solo dos fechas: esto
   permite comparar los cambios **año a año** (Fases 3.6, 4.2 y 5.3), además de la comparación
   acumulada de todo el rango. Con más periodos el notebook tarda más en ejecutarse (varios minutos
   adicionales en las Fases 3, 4 y 5) — es el costo esperado de un análisis multitemporal más denso
   y más apto para una tesis de grado.

## Fase 0. Configuración del entorno

Instalación de librerías y autenticación contra Google Earth Engine (GEE), la plataforma de acceso
abierto que usaremos como fuente de imágenes satelitales (objetivo específico 1).

### 0.1 Instalar y cargar librerías

In [ ]:
!pip install -q -U earthengine-api geemap

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from io import BytesIO
from PIL import Image
import os

print("Librerías cargadas correctamente.")

### 0.2 Autenticar y conectar con Google Earth Engine

Al ejecutar la siguiente celda se abrirá una ventana/enlace para iniciar sesión con tu cuenta de
Google y autorizar el acceso. Luego, reemplaza `EE_PROJECT_ID` por el **ID de tu proyecto de Google
Cloud** vinculado a Earth Engine (lo encuentras en https://code.earthengine.google.com/, arriba a la
izquierda, o en https://console.cloud.google.com/).

In [ ]:
EE_PROJECT_ID = "tu-proyecto-gee"  # <-- reemplaza con el ID de tu proyecto de Google Cloud / Earth Engine

ee.Authenticate()
ee.Initialize(project=EE_PROJECT_ID)

print("Conexión con Google Earth Engine establecida.")
print(f"Proyecto activo: {EE_PROJECT_ID}")

### 0.3 Carpeta de resultados (Google Drive)

Igual que en otros notebooks de este tipo, montamos Google Drive para guardar de forma persistente
las figuras, la tabla de estadísticas y el mapa de cambios generados en la Fase 5.

In [ ]:
from google.colab import drive

MONTAR_DRIVE = True  # pon False si no quieres usar Drive (los resultados solo quedarán en la sesión de Colab)

if MONTAR_DRIVE:
    drive.mount("/content/drive")
    CARPETA_RESULTADOS = "/content/drive/MyDrive/deteccion_cambios_bucaramanga"
else:
    CARPETA_RESULTADOS = "/content/deteccion_cambios_bucaramanga"

os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
print(f"Los resultados se guardarán en: {CARPETA_RESULTADOS}")

## Fase 1. Recopilación de imágenes satelitales multitemporales

**Objetivo específico:** *Recopilar un conjunto de imágenes satelitales multitemporales de una zona
geográfica específica, mediante el uso de plataformas de acceso abierto, como fuente de análisis de
los cambios en el terreno.*

**Plataforma de acceso abierto:** [Copernicus Sentinel-2](https://sentinel.esa.int/web/sentinel/missions/sentinel-2)
(programa espacial de la Unión Europea/ESA), distribuida a través de Google Earth Engine —
colección `COPERNICUS/S2_SR_HARMONIZED` (reflectancia de superficie, ya corregida
atmosféricamente, resolución de 10 m en las bandas visibles/infrarrojo cercano).

**Zona de estudio:** Área Metropolitana de Bucaramanga (AMB) — Bucaramanga, Floridablanca, Girón y Piedecuesta.

### 1.1 Definir el área de estudio (AMB)

In [ ]:
# Rectángulo delimitador del Área Metropolitana de Bucaramanga (AMB): cubre el casco urbano
# y la franja de expansión periurbana de los 4 municipios (Bucaramanga, Floridablanca, Girón
# y Piedecuesta). Es el AOI (Area Of Interest) principal usado en todo el notebook.
AMB_LON_MIN, AMB_LON_MAX = -73.22, -73.00
AMB_LAT_MIN, AMB_LAT_MAX = 6.93, 7.20

aoi_bbox = ee.Geometry.Rectangle(
    [AMB_LON_MIN, AMB_LAT_MIN, AMB_LON_MAX, AMB_LAT_MAX]
)

# Centro aproximado del AMB (Bucaramanga), usado para centrar los mapas interactivos.
AMB_CENTRO = [7.1193, -73.1227]

area_km2 = aoi_bbox.area().divide(1e6).getInfo()
print(f"Área de estudio (rectángulo delimitador del AMB): {area_km2:,.1f} km²")

### 1.2 (Opcional) Refinar el área con límites administrativos oficiales

Esta celda intenta reemplazar el rectángulo anterior por la **unión de los límites municipales
oficiales** (Bucaramanga, Floridablanca, Girón y Piedecuesta) tomados del conjunto de datos
`FAO/GAUL/2015/level2`. Si el conjunto de datos no está disponible, los nombres no coinciden, o el
resultado no pasa una **validación de tamaño** (para blindarnos contra una coincidencia de nombre
incorrecta que traiga un polígono lejano y agrande muchísimo el área), el notebook **conserva
automáticamente** el rectángulo delimitador de la celda anterior (`aoi_bbox`) — esta celda es
segura de ejecutar y no rompe el resto del flujo.

También se define `aoi_bounds`, el rectángulo delimitador **de la geometría final** (`aoi.bounds()`).
Se usa únicamente para descargar miniaturas/imágenes de vista previa (Fases 1.5, 1.6 y 5.2): así se
evita que, al recortar con un polígono irregular, la miniatura quede con grandes zonas en blanco
fuera del polígono pero dentro de su rectángulo — el análisis (Fase 2 en adelante) sigue usando el
polígono preciso `aoi`.

In [ ]:
MUNICIPIOS_AMB = ["Bucaramanga", "Floridablanca", "Giron", "Girón", "Piedecuesta"]

aoi = aoi_bbox  # valor por defecto: se sobrescribe abajo solo si el refinamiento funciona y pasa la validación
aoi_fuente = "Rectángulo delimitador (definido manualmente)"

# Rango de área aceptado para la geometría refinada, tomando como referencia el rectángulo de 1.1.
# Si el resultado de FAO GAUL cae fuera de este rango, es señal de una coincidencia de nombre
# incorrecta (p. ej. un municipio homónimo en otro lugar) y se descarta por seguridad.
FACTOR_AREA_MIN, FACTOR_AREA_MAX = 0.2, 4.0

try:
    limites_municipales = (
        ee.FeatureCollection("FAO/GAUL/2015/level2")
        .filter(ee.Filter.eq("ADM0_NAME", "Colombia"))
        .filter(ee.Filter.eq("ADM1_NAME", "Santander"))
        .filter(ee.Filter.inList("ADM2_NAME", MUNICIPIOS_AMB))
    )
    n_municipios = limites_municipales.size().getInfo()
    if n_municipios > 0:
        aoi_admin = limites_municipales.geometry().dissolve()
        area_admin_km2 = aoi_admin.area().divide(1e6).getInfo()
        razon_area = area_admin_km2 / area_km2

        if FACTOR_AREA_MIN <= razon_area <= FACTOR_AREA_MAX:
            aoi = aoi_admin
            aoi_fuente = f"Límites administrativos oficiales (FAO GAUL, {n_municipios} municipios encontrados)"
            print(f"AOI refinada con límites oficiales: {area_admin_km2:,.1f} km² (válida)")
        else:
            print(
                f"El área de FAO GAUL ({area_admin_km2:,.1f} km²) es sospechosamente distinta al "
                f"rectángulo delimitador ({area_km2:,.1f} km²) — probable coincidencia de nombre "
                "incorrecta. Se conserva el rectángulo delimitador."
            )
    else:
        print("No se encontraron los municipios por nombre en FAO GAUL; se conserva el rectángulo delimitador.")
except Exception as error:
    print(f"No fue posible refinar el AOI ({error}). Se conserva el rectángulo delimitador.")

aoi_bounds = aoi.bounds()

print(f"AOI activa (análisis): {aoi_fuente}")

### 1.3 Definir los periodos multitemporales de análisis

Definimos un **conjunto de periodos anuales** (no solo dos fechas) para cumplir con el enfoque
*multitemporal* y permitir una comparación **año a año**, no únicamente entre el primer y el
último año:

- La **comparación año a año** (Fases 3.6, 4.2 y 5.3) recorre cada **par de años consecutivos**
  (2017→2018, 2018→2019, …, 2023→2024) y produce un mapa de cambios independiente por cada
  intervalo, con la misma metodología — la base de la comparación interanual solicitada.
- La **comparación principal** (Fases 3.1-3.5 y 4.1/4.3/4.4) sigue siendo entre el primer y el
  último periodo (todo el rango de estudio), como resumen del cambio acumulado en 8 años.

Se eligen ventanas de **enero-marzo**, la temporada relativamente más seca del año en Bucaramanga,
para reducir la cobertura de nubes y mantener condiciones de observación comparables entre años.

In [ ]:
# Diccionario {etiqueta: (fecha_inicio, fecha_fin)}: un periodo anual (enero-marzo) por año, de
# 2017 (primer año con cobertura Sentinel-2 consistente sobre Colombia) a 2024. Puedes ajustar el
# rango o la densidad de AÑOS_ANALISIS; el resto del notebook se adapta automáticamente al primero
# y al último periodo, y a cada par de periodos consecutivos.
AÑOS_ANALISIS = list(range(2017, 2025))  # 2017, 2018, ..., 2024 -> 8 periodos, 7 comparaciones año a año

PERIODOS = {str(anio): (f"{anio}-01-01", f"{anio}-03-31") for anio in AÑOS_ANALISIS}

# Porcentaje máximo de nubosidad permitido por escena (metadato de la colección).
MAX_NUBES_ESCENA = 40

ETIQUETAS_PERIODOS = list(PERIODOS.keys())
T_INICIAL_LABEL = ETIQUETAS_PERIODOS[0]
T_FINAL_LABEL = ETIQUETAS_PERIODOS[-1]

# Pares de periodos consecutivos, p. ej. [("2017", "2018"), ("2018", "2019"), ..., ("2023", "2024")],
# usados en las Fases 3.6 / 4.2 / 5.3 para la comparación año a año.
PARES_CONSECUTIVOS = list(zip(ETIQUETAS_PERIODOS[:-1], ETIQUETAS_PERIODOS[1:]))

print(f"Periodos definidos ({len(ETIQUETAS_PERIODOS)}): {ETIQUETAS_PERIODOS}")
print(f"Comparación principal (todo el rango): {T_INICIAL_LABEL} → {T_FINAL_LABEL}")
print(f"Comparaciones año a año ({len(PARES_CONSECUTIVOS)}): {PARES_CONSECUTIVOS}")

### 1.4 Recopilar las colecciones Sentinel-2 para cada periodo

In [ ]:
COLECCION_S2 = "COPERNICUS/S2_SR_HARMONIZED"

COLECCIONES = {}

for etiqueta, (fecha_inicio, fecha_fin) in PERIODOS.items():
    coleccion = (
        ee.ImageCollection(COLECCION_S2)
        .filterBounds(aoi)
        .filterDate(fecha_inicio, fecha_fin)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_NUBES_ESCENA))
    )
    n_imagenes = coleccion.size().getInfo()
    COLECCIONES[etiqueta] = coleccion

    estado = "✅" if n_imagenes > 0 else "⚠️ sin imágenes: amplía el rango de fechas o el % de nubes"
    print(f"{etiqueta} ({fecha_inicio} a {fecha_fin}): {n_imagenes} imágenes encontradas {estado}")

### 1.5 Vista previa rápida (color verdadero) de un periodo

Una sola escena Sentinel-2 (una sola pasada del satélite) casi nunca cubre por completo un área
como el AMB: queda cortada por el borde de la franja de barrido, dejando huecos sin datos que se
ven como zonas "rotas" o vacías dentro del AOI. Para evitarlo, `obtener_mosaico_periodo` combina
**todas** las escenas disponibles del periodo en un mosaico (la menos nubosa queda arriba, y las
demás rellenan los huecos que esa escena no cubre) — sin promediar todavía nada, eso ocurre en la
Fase 2.

In [ ]:
def obtener_mosaico_periodo(coleccion):
    """Mosaico de todas las escenas del periodo: la escena menos nubosa queda visible arriba,
    y el resto rellena las zonas que esa escena no alcanza a cubrir (borde de la franja de
    barrido), garantizando cobertura completa del AOI."""
    return coleccion.sort("CLOUDY_PIXEL_PERCENTAGE", False).mosaic()

imagen_preliminar = obtener_mosaico_periodo(COLECCIONES[T_FINAL_LABEL])

vis_rgb_cruda = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}

mapa_previo = geemap.Map(center=AMB_CENTRO, zoom=11)
mapa_previo.addLayer(imagen_preliminar, vis_rgb_cruda, f"Sentinel-2 sin procesar ({T_FINAL_LABEL})")
mapa_previo.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": ["red"]}, "Límite del AOI (AMB)")
mapa_previo

### 1.6 Guardar en Google Drive el conjunto de imágenes recopiladas

Para poder **ver directamente en Drive** el conjunto de imágenes multitemporales recopilado (sin
depender del mapa interactivo), esta celda descarga una imagen en color verdadero por cada periodo
—el mosaico de escenas de la sección 1.5, con cobertura completa del AOI— y la guarda como PNG en
la carpeta ya montada en la Fase 0.3 (`CARPETA_RESULTADOS`), dentro de una subcarpeta
`01_imagenes_recopiladas`. La función `obtener_imagen_ee` se reutiliza más adelante en la Fase 5
para las figuras finales.

In [ ]:
def obtener_imagen_ee(imagen, vis_params, dimensiones=1024):
    """Descarga una imagen de Earth Engine como objeto PIL.Image, lista para guardar en disco o
    convertir a arreglo numpy para graficar. Usa `aoi_bounds` (rectángulo) en vez de `aoi`
    (que puede ser un polígono irregular) para que la miniatura salga completa, sin zonas en
    blanco fuera del polígono pero dentro de su rectángulo delimitador."""
    url = imagen.getThumbURL({**vis_params, "region": aoi_bounds, "dimensions": dimensiones, "format": "png"})
    respuesta = requests.get(url, timeout=60)
    respuesta.raise_for_status()
    return Image.open(BytesIO(respuesta.content))


CARPETA_IMAGENES_RECOPILADAS = os.path.join(CARPETA_RESULTADOS, "01_imagenes_recopiladas")
os.makedirs(CARPETA_IMAGENES_RECOPILADAS, exist_ok=True)

for etiqueta, coleccion in COLECCIONES.items():
    mosaico_periodo = obtener_mosaico_periodo(coleccion)
    n_escenas = coleccion.size().getInfo()

    imagen_png = obtener_imagen_ee(mosaico_periodo, vis_rgb_cruda)
    ruta_imagen = os.path.join(CARPETA_IMAGENES_RECOPILADAS, f"sentinel2_AMB_{etiqueta}.png")
    imagen_png.save(ruta_imagen)
    print(f"{etiqueta}: guardada {ruta_imagen} (mosaico de {n_escenas} escenas)")

print(f"\nConjunto de imágenes multitemporales guardado en Google Drive: {CARPETA_IMAGENES_RECOPILADAS}")

## Fase 2. Preprocesamiento de las imágenes satelitales

**Objetivo específico:** *Preprocesar las imágenes satelitales recopiladas, aplicando la técnica de
análisis multitemporal en la detección de cambios en el terreno, como expansión urbana,
deforestación o impacto de desastres naturales.*

Pasos: (2.1) enmascarado de nubes/sombras, (2.2) composición mediana libre de nubes por periodo,
(2.3) cálculo de índices espectrales (NDVI, NDBI, NDWI) que resumen, respectivamente, vegetación,
zonas construidas y cuerpos de agua.

### 2.1 Función de enmascarado de nubes y sombras (banda SCL)

In [ ]:
def enmascarar_nubes_sombras(imagen):
    """Enmascara nubes, sombras de nubes y cirros usando la banda SCL (Scene Classification)
    de Sentinel-2, y escala las bandas ópticas a reflectancia [0, 1]."""
    scl = imagen.select("SCL")
    # Clases SCL a excluir: 3 sombra de nube, 8 nube prob. media, 9 nube prob. alta, 10 cirro delgado.
    mascara_valida = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))

    bandas_opticas = imagen.select("B.*").multiply(0.0001)
    return (
        bandas_opticas.updateMask(mascara_valida)
        .copyProperties(imagen, imagen.propertyNames())
    )

print("Función de enmascarado de nubes definida.")

### 2.2 Generar el composite mediano (sin nubes) de cada periodo

In [ ]:
COMPOSITES = {}

for etiqueta, coleccion in COLECCIONES.items():
    composite = (
        coleccion
        .map(enmascarar_nubes_sombras)
        .median()
        .clip(aoi)
    )
    COMPOSITES[etiqueta] = composite

print(f"Composites generados para los periodos: {list(COMPOSITES.keys())}")

### 2.3 Cálculo de índices espectrales

In [ ]:
def calcular_indices(imagen):
    """Agrega al composite las bandas de índices espectrales:
    - NDVI  (B8, B4): vegetación
    - NDBI  (B11, B8): superficies construidas / suelo urbano
    - NDWI  (B3, B8): cuerpos de agua (McFeeters, 1996)
    """
    ndvi = imagen.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndbi = imagen.normalizedDifference(["B11", "B8"]).rename("NDBI")
    ndwi = imagen.normalizedDifference(["B3", "B8"]).rename("NDWI")
    return imagen.addBands([ndvi, ndbi, ndwi])

COMPOSITES = {etiqueta: calcular_indices(img) for etiqueta, img in COMPOSITES.items()}

print("Índices NDVI, NDBI y NDWI calculados para cada periodo.")
print("Bandas disponibles en cada composite:", COMPOSITES[T_FINAL_LABEL].bandNames().getInfo())

### 2.4 Verificación visual del preprocesamiento

In [ ]:
vis_rgb = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}
vis_ndvi = {"bands": ["NDVI"], "min": -0.2, "max": 0.8, "palette": ["#a50026", "#ffffbf", "#1a9850"]}

mapa_preprocesado = geemap.Map(center=AMB_CENTRO, zoom=11)
mapa_preprocesado.addLayer(COMPOSITES[T_INICIAL_LABEL], vis_rgb, f"Color verdadero {T_INICIAL_LABEL} (sin nubes)")
mapa_preprocesado.addLayer(COMPOSITES[T_FINAL_LABEL], vis_rgb, f"Color verdadero {T_FINAL_LABEL} (sin nubes)")
mapa_preprocesado.addLayer(COMPOSITES[T_INICIAL_LABEL], vis_ndvi, f"NDVI {T_INICIAL_LABEL}", shown=False)
mapa_preprocesado.addLayer(COMPOSITES[T_FINAL_LABEL], vis_ndvi, f"NDVI {T_FINAL_LABEL}", shown=False)
mapa_preprocesado.addLayerControl()
mapa_preprocesado

## Fase 3. Sistematización de la detección de cambios

**Objetivo específico:** *Sistematizar la técnica de detección de los cambios en la cobertura
terrestre, identificando las variaciones significativas en el terreno especificado.*

Metodología (reproducible, no basada en inspección visual subjetiva, y **empaquetada en funciones**
para poder aplicarse de forma idéntica a cualquier par de periodos):

1. Diferencia de índices espectrales entre dos periodos (ΔNDVI, ΔNDBI, ΔNDWI) — `calcular_deltas`.
2. **Umbral estadístico sistemático**: media ± *k* × desviación estándar de cada diferencia,
   calculado sobre el propio AOI — el mismo criterio se puede aplicar a cualquier otra zona o par
   de fechas sin ajuste manual — `obtener_umbrales`.
3. Combinación de los tres índices en un **mapa de cambios clasificado** con categorías
   interpretables (expansión urbana, pérdida/ganancia de vegetación, cambio hídrico, sin cambio) —
   `clasificar_cambios`.
4. Aplicación de `clasificar_cambios` tanto a la **comparación principal** (todo el rango de
   estudio) como a **cada par de años consecutivos** (Fase 3.6), para una comparación año a año.

### 3.1 Composites del periodo inicial y final

In [ ]:
imagen_inicial = COMPOSITES[T_INICIAL_LABEL]
imagen_final = COMPOSITES[T_FINAL_LABEL]

print(f"Periodo inicial: {T_INICIAL_LABEL} ({PERIODOS[T_INICIAL_LABEL][0]} a {PERIODOS[T_INICIAL_LABEL][1]})")
print(f"Periodo final:   {T_FINAL_LABEL} ({PERIODOS[T_FINAL_LABEL][0]} a {PERIODOS[T_FINAL_LABEL][1]})")

### 3.2 Función de diferencia de índices espectrales (ΔNDVI, ΔNDBI, ΔNDWI)

Se define como función porque se reutiliza en la Fase 3.6 para cada comparación año a año, no solo
para la comparación principal.

In [ ]:
def calcular_deltas(imagen_ini, imagen_fin):
    """Diferencias de índices espectrales entre dos composites cualesquiera: ΔNDVI, ΔNDBI y ΔNDWI.
    Se reutiliza tanto para la comparación principal como para cada comparación año a año (3.6)."""
    delta_ndvi = imagen_fin.select("NDVI").subtract(imagen_ini.select("NDVI")).rename("delta_NDVI")
    delta_ndbi = imagen_fin.select("NDBI").subtract(imagen_ini.select("NDBI")).rename("delta_NDBI")
    delta_ndwi = imagen_fin.select("NDWI").subtract(imagen_ini.select("NDWI")).rename("delta_NDWI")
    return delta_ndvi.addBands([delta_ndbi, delta_ndwi])

deltas = calcular_deltas(imagen_inicial, imagen_final)
print("Bandas de diferencia calculadas (comparación principal):", deltas.bandNames().getInfo())

### 3.3 Umbral estadístico sistemático (media ± k·desviación estándar)

In [ ]:
K_UMBRAL = 1.5  # sensibilidad del umbral; se explora en la Fase 4.4 (análisis de sensibilidad)

def obtener_umbrales(imagen_delta, banda, aoi, escala=20, k=K_UMBRAL):
    """Calcula (umbral_inferior, umbral_superior) = media ± k·desv. estándar de una banda
    de diferencia, sobre el AOI. Es el mismo criterio estadístico para cualquier banda, zona o
    par de periodos: se reutiliza para la comparación principal y para cada año a año (3.6)."""
    estadisticas = imagen_delta.select(banda).reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=aoi,
        scale=escala,
        bestEffort=True,
        maxPixels=1e9,
    ).getInfo()

    media = estadisticas[f"{banda}_mean"]
    desviacion = estadisticas[f"{banda}_stdDev"]
    return media - k * desviacion, media + k * desviacion

umbral_ndvi_inf, umbral_ndvi_sup = obtener_umbrales(deltas, "delta_NDVI", aoi)
umbral_ndbi_inf, umbral_ndbi_sup = obtener_umbrales(deltas, "delta_NDBI", aoi)
umbral_ndwi_inf, umbral_ndwi_sup = obtener_umbrales(deltas, "delta_NDWI", aoi)

print(f"Umbral ΔNDVI: [{umbral_ndvi_inf:.4f}, {umbral_ndvi_sup:.4f}]  (pérdida de vegetación / ganancia de vegetación)")
print(f"Umbral ΔNDBI: [{umbral_ndbi_inf:.4f}, {umbral_ndbi_sup:.4f}]  (expansión urbana)")
print(f"Umbral ΔNDWI: [{umbral_ndwi_inf:.4f}, {umbral_ndwi_sup:.4f}]  (cambio hídrico)")

### 3.4 Análisis de Vector de Cambio (CVA) — magnitud del cambio

Complementa el análisis por índices individuales con una medida conjunta de "cuánto" cambió cada
píxel, combinando vegetación (NDVI) y superficie construida (NDBI) en un solo vector.

In [ ]:
magnitud_cambio = (
    deltas.select("delta_NDVI").pow(2)
    .add(deltas.select("delta_NDBI").pow(2))
    .sqrt()
    .rename("magnitud_CVA")
)

umbral_magnitud = magnitud_cambio.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=aoi, scale=20, bestEffort=True, maxPixels=1e9,
).getInfo()

umbral_cva = umbral_magnitud["magnitud_CVA_mean"] + K_UMBRAL * umbral_magnitud["magnitud_CVA_stdDev"]
print(f"Umbral de magnitud de cambio (CVA): {umbral_cva:.4f} — píxeles por encima se consideran 'cambio relevante'")

### 3.5 Función de clasificación de cambios (y mapa de la comparación principal)

Se combina lo anterior en una **función reutilizable** (`clasificar_cambios`) que, dado cualquier
par de composites, calcula sus diferencias, obtiene los umbrales estadísticos y construye el mapa
de cambios clasificado en 5 categorías mutuamente excluyentes, aplicando los umbrales en orden de
prioridad (agua → vegetación → urbano → sin cambio):

| Código | Categoría | Criterio |
|---|---|---|
| 0 | Sin cambio significativo | ninguno de los criterios siguientes se cumple |
| 1 | Pérdida de vegetación (posible deforestación) | ΔNDVI < umbral inferior |
| 2 | Ganancia de vegetación | ΔNDVI > umbral superior |
| 3 | Expansión urbana / suelo construido | ΔNDBI > umbral superior |
| 4 | Cambio en cuerpos de agua | \|ΔNDWI\| > umbral superior |

Se aplica aquí a la **comparación principal** (todo el rango de estudio) y se reutiliza en la
Fase 3.6 para cada comparación año a año — garantizando que se aplique **exactamente el mismo
criterio metodológico** en ambos casos.

In [ ]:
CATEGORIAS_CAMBIO = {
    0: ("Sin cambio significativo", "#d9d9d9"),
    1: ("Pérdida de vegetación (posible deforestación)", "#d73027"),
    2: ("Ganancia de vegetación", "#1a9850"),
    3: ("Expansión urbana / suelo construido", "#fdae61"),
    4: ("Cambio en cuerpos de agua", "#4575b4"),
}
paleta_cambios = [color for _, color in CATEGORIAS_CAMBIO.values()]


def clasificar_cambios(imagen_ini, imagen_fin, aoi, k=K_UMBRAL, escala=20):
    """Aplica la metodología sistematizada de la Fase 3 a cualquier par de composites: calcula
    las diferencias de índices, obtiene los umbrales estadísticos (media ± k·σ) y construye el
    mapa de cambios clasificado en 5 categorías. Devuelve (mapa_clasificado, umbrales) para poder
    inspeccionar los umbrales usados en cada comparación."""
    deltas_par = calcular_deltas(imagen_ini, imagen_fin)

    umb_ndvi = obtener_umbrales(deltas_par, "delta_NDVI", aoi, escala=escala, k=k)
    umb_ndbi = obtener_umbrales(deltas_par, "delta_NDBI", aoi, escala=escala, k=k)
    umb_ndwi = obtener_umbrales(deltas_par, "delta_NDWI", aoi, escala=escala, k=k)

    mapa = (
        ee.Image(0)
        .where(deltas_par.select("delta_NDVI").gt(umb_ndvi[1]), 2)   # ganancia de vegetación
        .where(deltas_par.select("delta_NDVI").lt(umb_ndvi[0]), 1)   # pérdida de vegetación
        .where(deltas_par.select("delta_NDBI").gt(umb_ndbi[1]), 3)   # expansión urbana
        .where(deltas_par.select("delta_NDWI").abs().gt(umb_ndwi[1]), 4)  # cambio hídrico
        .rename("clase_cambio")
        .clip(aoi)
        .updateMask(imagen_ini.select("NDVI").mask().And(imagen_fin.select("NDVI").mask()))
    )
    return mapa, {"NDVI": umb_ndvi, "NDBI": umb_ndbi, "NDWI": umb_ndwi}


mapa_cambios, umbrales_principal = clasificar_cambios(imagen_inicial, imagen_final, aoi)

mapa_clasificado_vis = geemap.Map(center=AMB_CENTRO, zoom=11)
mapa_clasificado_vis.addLayer(
    mapa_cambios,
    {"min": 0, "max": 4, "palette": paleta_cambios},
    f"Cambios {T_INICIAL_LABEL} → {T_FINAL_LABEL}",
)
mapa_clasificado_vis.add_legend(
    title="Categoría de cambio",
    legend_dict={nombre: color for nombre, color in CATEGORIAS_CAMBIO.values()},
)
mapa_clasificado_vis

### 3.6 Detección de cambios año a año (comparaciones consecutivas)

Para lograr una **comparación exitosa** entre lo ocurrido en cada año —y no solo entre el primero y
el último—, se aplica `clasificar_cambios` a **cada par de periodos consecutivos** definido en
`PARES_CONSECUTIVOS` (Fase 1.3): 2017→2018, 2018→2019, …, 2023→2024. El resultado es un mapa de
cambios independiente por cada intervalo de un año, calculado con **exactamente la misma
metodología** (mismos criterios, umbrales recalculados para cada par) que la comparación principal.

In [ ]:
MAPAS_CAMBIO_ANUAL = {}
UMBRALES_ANUALES = {}

for anio_ini, anio_fin in PARES_CONSECUTIVOS:
    par_label = f"{anio_ini}→{anio_fin}"
    mapa_par, umbrales_par = clasificar_cambios(COMPOSITES[anio_ini], COMPOSITES[anio_fin], aoi)
    MAPAS_CAMBIO_ANUAL[par_label] = mapa_par
    UMBRALES_ANUALES[par_label] = umbrales_par
    print(f"{par_label}: mapa de cambios calculado (umbral ΔNDBI = {umbrales_par['NDBI'][1]:.4f})")

print(f"\n{len(MAPAS_CAMBIO_ANUAL)} mapas de cambio año a año calculados.")

## Fase 4. Evaluación y validación de los resultados

**Objetivo específico:** *Evaluar los resultados obtenidos, validando la confiabilidad de la
herramienta propuesta, mediante el análisis comparativo de las imágenes satelitales multitemporales.*

Se valida el mapa de cambios de cuatro formas complementarias:
1. Estadísticas de área por categoría — comparación principal (4.1) y año a año (4.2).
2. Validación cruzada **multiclase** con un conjunto de datos independiente (Dynamic World de
   Google): matriz de confusión, **OA, AA y Kappa** (4.3) — el mismo esquema de métricas que se
   reporta habitualmente al validar clasificadores de cobertura de suelo sobre conjuntos de
   referencia académicos como **Indian Pines**.
3. Análisis de sensibilidad del umbral estadístico (robustez del método ante *k*) (4.4).

### 4.1 Estadísticas de área por categoría de cambio (comparación principal)

In [ ]:
def calcular_area_por_categoria(mapa_clases, aoi, escala=20):
    """Área (ha) de cada categoría de un mapa de cambios clasificado, agregada sobre el AOI.
    Función reutilizable: se aplica aquí a la comparación principal y, en la Fase 4.2, a cada
    comparación año a año."""
    area_pixel = ee.Image.pixelArea().divide(10000)  # hectáreas por píxel
    resultado = (
        area_pixel.addBands(mapa_clases)
        .reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName="clase"),
            geometry=aoi, scale=escala, bestEffort=True, maxPixels=1e9,
        )
        .getInfo()
    )
    filas = []
    for grupo in resultado["groups"]:
        clase = int(grupo["clase"])
        nombre, color = CATEGORIAS_CAMBIO[clase]
        filas.append({"clase": clase, "categoria": nombre, "hectareas": grupo["sum"], "color": color})
    return pd.DataFrame(filas).sort_values("clase").reset_index(drop=True)


tabla_area = calcular_area_por_categoria(mapa_cambios, aoi)
tabla_area["porcentaje"] = 100 * tabla_area["hectareas"] / tabla_area["hectareas"].sum()

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")
tabla_area[["categoria", "hectareas", "porcentaje"]]

### 4.2 Estadísticas de área por año (comparaciones consecutivas)

Aplica `calcular_area_por_categoria` (4.1) a cada uno de los mapas de cambio año a año de la
Fase 3.6, produciendo una tabla larga (`periodo`, `categoria`, `hectareas`) que alimenta la
visualización de evolución interanual de la Fase 5.5.

In [ ]:
filas_anuales = []
for par_label, mapa_par in MAPAS_CAMBIO_ANUAL.items():
    tabla_par = calcular_area_por_categoria(mapa_par, aoi)
    tabla_par["periodo"] = par_label
    filas_anuales.append(tabla_par)

tabla_area_anual = pd.concat(filas_anuales, ignore_index=True)
tabla_area_anual[["periodo", "categoria", "hectareas"]]

### 4.3 Validación cruzada multiclase (matriz de confusión, OA, AA y Kappa)

No se cuenta con datos de verdad de campo, por lo que la validación se hace comparando el mapa de
cambios propio contra un producto de cobertura de suelo **independiente y ya publicado**:
[Dynamic World](https://dynamicworld.app/) (Google/WRI), que ofrece probabilidades de cobertura
(`built`, `trees`, `grass`, `crops`, `shrub_and_scrub`, `water`, …) casi en tiempo real desde 2015,
a 10 m de resolución. A partir de esas probabilidades se construye un mapa de cambios de
**referencia** con las mismas 5 categorías que nuestro mapa propio (Fase 3.5), y se comparan ambos
con las métricas estándar de evaluación de clasificaciones en teledetección — el mismo esquema que
se usa, por ejemplo, al validar clasificadores sobre el conjunto de referencia **Indian Pines**:

- **Matriz de confusión** (una fila/columna por categoría presente en la muestra).
- **OA (Overall Accuracy / exactitud global)**: proporción total de puntos de muestra coincidentes.
- **AA (Average Accuracy / exactitud promedio)**: promedio de la exactitud (sensibilidad) de cada
  categoría por separado — más exigente que la OA cuando hay categorías minoritarias, como suele
  ocurrir con "cambio en cuerpos de agua" frente a "sin cambio".
- **Índice Kappa de Cohen**: concordancia corregida por azar.

El muestreo es **aleatorio estratificado** (`stratifiedSample`, la misma técnica usada para separar
puntos de entrenamiento/prueba de forma balanceada en trabajos de clasificación de imágenes): se
toman puntos por igual de cada una de las 5 categorías, para que las clases minoritarias no queden
subrepresentadas.

In [ ]:
def obtener_prob_dw(banda, fecha_inicio, fecha_fin, aoi):
    """Promedio de una banda de probabilidad de Dynamic World (0-1) sobre un periodo y el AOI."""
    return (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterBounds(aoi)
        .filterDate(fecha_inicio, fecha_fin)
        .select(banda)
        .median()
        .clip(aoi)
    )

# Probabilidad de vegetación combinada (árboles + pastos + cultivos + arbustos) para el periodo
# inicial y final de la comparación principal; "built" y "water" se usan directamente.
BANDAS_VEGETACION_DW = ["trees", "grass", "crops", "shrub_and_scrub"]

def obtener_vegetacion_dw(fecha_inicio, fecha_fin, aoi):
    bandas = [obtener_prob_dw(b, fecha_inicio, fecha_fin, aoi) for b in BANDAS_VEGETACION_DW]
    return bandas[0].add(bandas[1]).add(bandas[2]).add(bandas[3]).rename("vegetacion")

veg_inicial_dw = obtener_vegetacion_dw(*PERIODOS[T_INICIAL_LABEL], aoi)
veg_final_dw = obtener_vegetacion_dw(*PERIODOS[T_FINAL_LABEL], aoi)
built_inicial_dw = obtener_prob_dw("built", *PERIODOS[T_INICIAL_LABEL], aoi)
built_final_dw = obtener_prob_dw("built", *PERIODOS[T_FINAL_LABEL], aoi)
water_inicial_dw = obtener_prob_dw("water", *PERIODOS[T_INICIAL_LABEL], aoi)
water_final_dw = obtener_prob_dw("water", *PERIODOS[T_FINAL_LABEL], aoi)

delta_veg_dw = veg_final_dw.subtract(veg_inicial_dw)
delta_built_dw = built_final_dw.subtract(built_inicial_dw)
delta_water_dw = water_final_dw.subtract(water_inicial_dw)

# Umbral de cambio mínimo en probabilidad de Dynamic World (0-1) para considerarlo relevante.
UMBRAL_DW = 0.10

mapa_cambios_dw = (
    ee.Image(0)
    .where(delta_veg_dw.gt(UMBRAL_DW), 2)
    .where(delta_veg_dw.lt(-UMBRAL_DW), 1)
    .where(delta_built_dw.gt(UMBRAL_DW), 3)
    .where(delta_water_dw.abs().gt(UMBRAL_DW), 4)
    .rename("clase_cambio_dw")
)

print("Mapa de cambios de referencia (Dynamic World) construido con las mismas 5 categorías.")

Muestreo aleatorio estratificado y cálculo de las métricas de validación:

In [ ]:
from sklearn.metrics import confusion_matrix, cohen_kappa_score, classification_report

NOMBRES_CLASES = [CATEGORIAS_CAMBIO[c][0] for c in range(5)]

capas_comparacion = mapa_cambios.rename("propio").addBands(mapa_cambios_dw.rename("referencia"))

muestra = capas_comparacion.stratifiedSample(
    numPoints=80,          # puntos por categoría (muestreo balanceado; hasta 5 x 80 = 400 puntos)
    classBand="propio",
    region=aoi,
    scale=20,
    seed=42,
    geometries=False,
).getInfo()

registros = [f["properties"] for f in muestra["features"]]
df_muestra = pd.DataFrame(registros).dropna()

y_propio = df_muestra["propio"].astype(int)
y_referencia = df_muestra["referencia"].astype(int)

etiquetas_presentes = sorted(set(y_referencia) | set(y_propio))
matriz_confusion = confusion_matrix(y_referencia, y_propio, labels=etiquetas_presentes)

oa = (y_referencia == y_propio).mean()
kappa = cohen_kappa_score(y_referencia, y_propio)
exactitud_por_clase = matriz_confusion.diagonal() / matriz_confusion.sum(axis=1).clip(min=1)
aa = exactitud_por_clase.mean()

print(f"Tamaño de la muestra evaluada: {len(df_muestra)} puntos ({len(etiquetas_presentes)} categorías presentes)")
print(f"OA  (Overall Accuracy):  {oa:.1%}")
print(f"AA  (Average Accuracy):  {aa:.1%}")
print(f"Kappa de Cohen:          {kappa:.3f}")

print("\nReporte de clasificación (precisión / sensibilidad / F1 por categoría):")
print(classification_report(
    y_referencia, y_propio,
    labels=etiquetas_presentes,
    target_names=[NOMBRES_CLASES[c] for c in etiquetas_presentes],
    zero_division=0,
))

Matriz de confusión normalizada (proporción de cada categoría de referencia clasificada en cada categoría propia):

In [ ]:
matriz_normalizada = matriz_confusion / matriz_confusion.sum(axis=1, keepdims=True).clip(min=1)
etiquetas_clase = [NOMBRES_CLASES[c] for c in etiquetas_presentes]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(matriz_normalizada, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(etiquetas_clase)))
ax.set_yticks(range(len(etiquetas_clase)))
ax.set_xticklabels(etiquetas_clase, rotation=40, ha="right")
ax.set_yticklabels(etiquetas_clase)
ax.set_xlabel("Categoría — mapa propio")
ax.set_ylabel("Categoría — referencia (Dynamic World)")
ax.set_title(f"Matriz de confusión normalizada (OA={oa:.1%}, AA={aa:.1%}, Kappa={kappa:.3f})")

for i in range(len(etiquetas_clase)):
    for j in range(len(etiquetas_clase)):
        color_texto = "white" if matriz_normalizada[i, j] > 0.5 else "black"
        ax.text(j, i, f"{matriz_normalizada[i, j]:.0%}", ha="center", va="center", color=color_texto, fontsize=9)

fig.colorbar(im, ax=ax, label="Proporción dentro de cada categoría de referencia")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "matriz_confusion.png"), dpi=150)
plt.show()

### 4.4 Análisis de sensibilidad del umbral (robustez del método)

Se recalcula el porcentaje de área con cambios significativos usando distintos valores de *k*
(multiplicador de la desviación estándar) para verificar que la herramienta no depende de forma
crítica de un único valor arbitrario.

In [ ]:
valores_k = [1.0, 1.25, 1.5, 1.75, 2.0]
resultados_sensibilidad = []

for k in valores_k:
    umb_ndvi_inf, umb_ndvi_sup = obtener_umbrales(deltas, "delta_NDVI", aoi, k=k)
    umb_ndbi_inf, umb_ndbi_sup = obtener_umbrales(deltas, "delta_NDBI", aoi, k=k)

    mapa_k = (
        ee.Image(0)
        .where(deltas.select("delta_NDVI").gt(umb_ndvi_sup), 2)
        .where(deltas.select("delta_NDVI").lt(umb_ndvi_inf), 1)
        .where(deltas.select("delta_NDBI").gt(umb_ndbi_sup), 3)
    )

    area_cambio_km2 = (
        ee.Image.pixelArea().divide(1e6)
        .updateMask(mapa_k.gt(0))
        .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi, scale=20, bestEffort=True, maxPixels=1e9)
        .getInfo()
    )
    km2 = area_cambio_km2.get("area", 0)
    resultados_sensibilidad.append({"k": k, "area_cambio_km2": km2, "pct_area_total": 100 * km2 / area_km2})

tabla_sensibilidad = pd.DataFrame(resultados_sensibilidad)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tabla_sensibilidad["k"], tabla_sensibilidad["pct_area_total"], marker="o", color="#2166ac")
ax.set_xlabel("k (umbral = media ± k·desviación estándar)")
ax.set_ylabel("% del AOI clasificado como cambio")
ax.set_title("Análisis de sensibilidad del umbral estadístico")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "sensibilidad_umbral.png"), dpi=150)
plt.show()

tabla_sensibilidad

## Fase 5. Visualización de los resultados

**Objetivo específico:** *Generar visualizaciones de los cambios detectados, empleando librerías
de representación gráfica, permitiendo la interpretación de los cambios detectados.*

### 5.1 Mapa interactivo comparativo (antes / después / cambios)

In [ ]:
mapa_final = geemap.Map(center=AMB_CENTRO, zoom=11)
mapa_final.addLayer(imagen_inicial, vis_rgb, f"Color verdadero {T_INICIAL_LABEL}", shown=False)
mapa_final.addLayer(imagen_final, vis_rgb, f"Color verdadero {T_FINAL_LABEL}", shown=True)
mapa_final.addLayer(
    mapa_cambios,
    {"min": 0, "max": 4, "palette": paleta_cambios},
    f"Mapa de cambios {T_INICIAL_LABEL} → {T_FINAL_LABEL}",
)
mapa_final.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": ["black"]}, "Límite del AOI (AMB)")
mapa_final.add_legend(
    title="Categoría de cambio",
    legend_dict={nombre: color for nombre, color in CATEGORIAS_CAMBIO.values()},
)
mapa_final.addLayerControl()
mapa_final

### 5.2 Comparación visual estática (antes / después / cambios) — comparación principal

Reutiliza `obtener_imagen_ee` (definida en la Fase 1.6) para descargar las miniaturas como arreglos
numpy y componer la figura comparativa del periodo completo de estudio (primer año → último año).

In [ ]:
img_inicial_arr = np.array(obtener_imagen_ee(imagen_inicial, vis_rgb))
img_final_arr = np.array(obtener_imagen_ee(imagen_final, vis_rgb))
img_cambios_arr = np.array(obtener_imagen_ee(mapa_cambios, {"min": 0, "max": 4, "palette": paleta_cambios}))

fig, ejes = plt.subplots(1, 3, figsize=(16, 5.5))

ejes[0].imshow(img_inicial_arr)
ejes[0].set_title(f"Área Metropolitana de Bucaramanga — {T_INICIAL_LABEL}")
ejes[0].axis("off")

ejes[1].imshow(img_final_arr)
ejes[1].set_title(f"Área Metropolitana de Bucaramanga — {T_FINAL_LABEL}")
ejes[1].axis("off")

ejes[2].imshow(img_cambios_arr)
ejes[2].set_title(f"Cambios detectados ({T_INICIAL_LABEL} → {T_FINAL_LABEL})")
ejes[2].axis("off")

parches_leyenda = [
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=color, markersize=12, label=nombre)
    for nombre, color in CATEGORIAS_CAMBIO.values()
]
fig.legend(handles=parches_leyenda, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "comparacion_antes_despues_cambios.png"), dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Panel de mapas de cambio año a año

Un mosaico de mapas —uno por cada comparación consecutiva de la Fase 3.6— para **comparar
visualmente cómo cambió el AMB año a año**, en vez de ver solo el resultado acumulado de todo el
periodo de estudio. Es la visualización central para responder "¿qué cambió, y en qué año?".

In [ ]:
n_pares = len(MAPAS_CAMBIO_ANUAL)
n_columnas = min(4, n_pares)
n_filas = -(-n_pares // n_columnas)  # división entera hacia arriba

fig, ejes = plt.subplots(n_filas, n_columnas, figsize=(4.2 * n_columnas, 4.6 * n_filas))
ejes = np.atleast_1d(ejes).flatten()

for eje, (par_label, mapa_par) in zip(ejes, MAPAS_CAMBIO_ANUAL.items()):
    arreglo = np.array(obtener_imagen_ee(mapa_par, {"min": 0, "max": 4, "palette": paleta_cambios}, dimensiones=512))
    eje.imshow(arreglo)
    eje.set_title(par_label, fontsize=11)
    eje.axis("off")

for eje_sobrante in ejes[n_pares:]:
    eje_sobrante.axis("off")

parches_leyenda = [
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=color, markersize=12, label=nombre)
    for nombre, color in CATEGORIAS_CAMBIO.values()
]
fig.legend(handles=parches_leyenda, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle("Mapas de cambio año a año — Área Metropolitana de Bucaramanga", fontsize=14, y=1.02)

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "mapas_cambio_anual.png"), dpi=150, bbox_inches="tight")
plt.show()

### 5.4 Serie de tiempo de NDVI y NDBI promedio en el AMB (monitoreo multitemporal)

In [ ]:
registros_series = []
for etiqueta in ETIQUETAS_PERIODOS:
    promedios = COMPOSITES[etiqueta].select(["NDVI", "NDBI"]).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=30, bestEffort=True, maxPixels=1e9,
    ).getInfo()
    registros_series.append({"periodo": etiqueta, "NDVI_promedio": promedios["NDVI"], "NDBI_promedio": promedios["NDBI"]})

serie_tiempo = pd.DataFrame(registros_series)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(serie_tiempo["periodo"], serie_tiempo["NDVI_promedio"], marker="o", color="#1a9850", label="NDVI promedio (vegetación)")
ax.plot(serie_tiempo["periodo"], serie_tiempo["NDBI_promedio"], marker="s", color="#fdae61", label="NDBI promedio (suelo construido)")
ax.set_xlabel("Periodo")
ax.set_ylabel("Valor promedio del índice")
ax.set_title("Evolución multitemporal de NDVI y NDBI en el AMB")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "serie_tiempo_ndvi_ndbi.png"), dpi=150)
plt.show()

serie_tiempo

### 5.5 Evolución del área por categoría a través de los años

A partir de `tabla_area_anual` (Fase 4.2), se grafica cómo cambió el área (hectáreas) de cada
categoría en cada intervalo año a año — la comparación interanual que permite identificar
tendencias (p. ej. si la expansión urbana se acelera o si la pérdida de vegetación es constante),
en lugar de un único número acumulado para todo el periodo de estudio.

In [ ]:
tabla_pivote = (
    tabla_area_anual[tabla_area_anual["clase"] != 0]
    .pivot(index="periodo", columns="categoria", values="hectareas")
    .reindex([f"{a}→{b}" for a, b in PARES_CONSECUTIVOS])
    .fillna(0)
)

colores_categoria = {nombre: color for nombre, color in CATEGORIAS_CAMBIO.values() if nombre != CATEGORIAS_CAMBIO[0][0]}

fig, ax = plt.subplots(figsize=(10, 5.5))
tabla_pivote.plot(kind="bar", ax=ax, color=[colores_categoria.get(col, "#999999") for col in tabla_pivote.columns])
ax.set_xlabel("Comparación año a año")
ax.set_ylabel("Área (hectáreas)")
ax.set_title("Evolución interanual del área por categoría de cambio — AMB")
ax.legend(title="Categoría", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "evolucion_area_anual.png"), dpi=150, bbox_inches="tight")
plt.show()

tabla_pivote

### 5.6 Gráfico de área por categoría de cambio (comparación principal)

In [ ]:
tabla_area_graf = tabla_area[tabla_area["clase"] != 0].sort_values("hectareas", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
barras = ax.barh(tabla_area_graf["categoria"], tabla_area_graf["hectareas"], color=tabla_area_graf["color"])
ax.set_xlabel("Área (hectáreas)")
ax.set_title(f"Área por categoría de cambio — AMB ({T_INICIAL_LABEL} → {T_FINAL_LABEL})")
for barra, valor in zip(barras, tabla_area_graf["hectareas"]):
    ax.text(valor, barra.get_y() + barra.get_height() / 2, f" {valor:,.0f} ha", va="center")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "area_por_categoria.png"), dpi=150)
plt.show()

### 5.7 Exportar resultados

Se guardan/descargan los productos finales:
1. **Tablas de estadísticas**: `estadisticas_cambio_principal.csv` (Fase 4.1) y
   `estadisticas_cambio_anual.csv` (Fase 4.2, año a año).
2. **Figuras** (PNG) generadas en 4.3, 4.4, 5.2, 5.3, 5.5 y 5.6 — ya guardadas en `CARPETA_RESULTADOS`.
3. **Mapa de cambios clasificado de la comparación principal** (GeoTIFF) — exportado a Google Drive
   mediante una tarea de Earth Engine (puede tardar algunos minutos; revisa el progreso en la
   pestaña *Tasks* de https://code.earthengine.google.com/). Los mapas año a año ya quedaron
   guardados como imagen (PNG) en el panel de la Fase 5.3; si necesitas cada uno como GeoTIFF por
   separado para un SIG, repite este mismo patrón de exportación dentro de un bucle sobre
   `MAPAS_CAMBIO_ANUAL`.

In [ ]:
ruta_csv = os.path.join(CARPETA_RESULTADOS, "estadisticas_cambio_principal.csv")
tabla_area.to_csv(ruta_csv, index=False)
print(f"Tabla de estadísticas (comparación principal) guardada en: {ruta_csv}")

ruta_csv_anual = os.path.join(CARPETA_RESULTADOS, "estadisticas_cambio_anual.csv")
tabla_area_anual.to_csv(ruta_csv_anual, index=False)
print(f"Tabla de estadísticas (año a año) guardada en: {ruta_csv_anual}")

tarea_exportacion = ee.batch.Export.image.toDrive(
    image=mapa_cambios.toByte(),
    description="mapa_cambios_AMB",
    folder="deteccion_cambios_bucaramanga",
    fileNamePrefix=f"mapa_cambios_{T_INICIAL_LABEL}_{T_FINAL_LABEL}",
    region=aoi,
    scale=10,
    maxPixels=1e9,
)
tarea_exportacion.start()
print("Tarea de exportación del mapa de cambios principal (GeoTIFF) iniciada.")
print("Revisa su progreso en la pestaña 'Tasks' de https://code.earthengine.google.com/")

from google.colab import files
files.download(ruta_csv)
files.download(ruta_csv_anual)

## Conclusiones y próximos pasos

Este notebook cubrió, fase a fase, los cinco objetivos específicos del proyecto para el Área
Metropolitana de Bucaramanga, con un diseño pensado para sostener un trabajo de investigación
académica (tesis de grado):

1. **Recopilación** — 8 periodos anuales (2017-2024) de imágenes Sentinel-2 de acceso abierto
   (Google Earth Engine), mosaicados por periodo para garantizar cobertura completa del AOI.
2. **Preprocesamiento** — Enmascarado de nubes, composición mediana e índices espectrales (NDVI, NDBI, NDWI).
3. **Sistematización** — Diferencias de índices con umbral estadístico reproducible, empaquetadas en
   funciones reutilizables (`calcular_deltas`, `obtener_umbrales`, `clasificar_cambios`) y aplicadas
   tanto a la comparación principal (todo el rango) como a **cada par de años consecutivos**.
4. **Evaluación** — Estadísticas de área (principal y año a año), validación cruzada **multiclase**
   con Dynamic World (matriz de confusión, **OA, AA y Kappa** — el mismo esquema de métricas que se
   reporta en benchmarks de clasificación de cobertura como **Indian Pines**) y análisis de
   sensibilidad del umbral.
5. **Visualización** — Mapa interactivo, comparación antes/después, **panel de mapas de cambio año a
   año**, serie de tiempo, **evolución interanual del área por categoría** y exportación de resultados.

**Posibles extensiones:**
- Incorporar imágenes de mayor resolución (p. ej. PlanetScope) para detectar cambios más finos.
- Aumentar la densidad temporal (semestral o trimestral) modificando `AÑOS_ANALISIS`/`PERIODOS`
  (Fase 1.3) para un monitoreo aún más continuo.
- Sustituir la clasificación basada en umbrales por un clasificador supervisado (Random Forest /
  redes neuronales) entrenado con puntos de verdad de campo, si llegan a estar disponibles — la
  validación de la Fase 4.3 (OA/AA/Kappa/matriz de confusión) seguiría siendo aplicable sin cambios.
- Aplicar la misma metodología a otras zonas cambiando únicamente `aoi_bbox` y `MUNICIPIOS_AMB` (Fase 1).

## Solución de problemas

- **`EEException: Not signed up for Earth Engine` o error de autenticación**: crea/activa tu cuenta en
  https://code.earthengine.google.com/register y verifica que `EE_PROJECT_ID` (Fase 0.2) sea el ID
  correcto de tu proyecto de Google Cloud.
- **`0 imágenes encontradas` en algún periodo (Fase 1.4)**: amplía el rango de fechas del periodo en
  `PERIODOS` o aumenta `MAX_NUBES_ESCENA`; Bucaramanga tiene temporadas con alta nubosidad.
- **`Computation timed out` o `Too many pixels in the region`**: reduce el parámetro `scale` en las
  llamadas a `reduceRegion` (por ejemplo de 10 a 20 o 30 m) o reduce el tamaño del AOI.
- **La celda 1.2 imprime "No fue posible refinar el AOI"**: no es un error crítico — el notebook
  sigue funcionando con el rectángulo delimitador definido en la celda 1.1.
- **La imagen se ve "rota", cortada en diagonal, o con partes del AOI en blanco (Fase 1.5/1.6)**:
  es normal en una sola escena Sentinel-2, que no siempre cubre el AOI completo (borde de la franja
  de barrido del satélite). Por eso estas celdas usan `obtener_mosaico_periodo` (combina varias
  escenas) en vez de una sola imagen; si el hueco persiste, ese periodo tiene muy pocas escenas
  disponibles — amplía su rango de fechas en `PERIODOS` (1.3) o sube `MAX_NUBES_ESCENA` (1.4).
- **La descarga de miniaturas (`obtener_imagen_ee`, Fases 1.6, 5.2 y 5.3) falla o tarda mucho**:
  revisa tu conexión a internet; si el AOI es muy grande, reduce el parámetro `dimensiones`.
- **Las Fases 3.6, 4.2 y 5.3 tardan varios minutos**: es normal — con 8 periodos anuales el
  notebook recalcula la metodología completa 7 veces (una por cada par de años consecutivos). Si
  quieres una ejecución más rápida (a costa de menos detalle año a año), reduce `AÑOS_ANALISIS`
  (Fase 1.3) a menos años.
- **El Kappa/OA/AA de la Fase 4.3 salen bajos**: es esperable cierto desacuerdo, ya que Dynamic
  World y el método propio usan sensores/criterios distintos; revisa `UMBRAL_DW` y `K_UMBRAL` — el
  análisis de sensibilidad (4.4) ayuda a elegir un valor de `K_UMBRAL` más estable.
- **`classification_report` o la matriz de confusión de la Fase 4.3 no incluyen las 5 categorías**:
  ocurre si alguna categoría (p. ej. "cambio en cuerpos de agua") no aparece en la muestra aleatoria
  del AOI; el código ya filtra dinámicamente por `etiquetas_presentes`, así que no es un error —
  simplemente esa categoría es poco frecuente en el periodo/zona analizados.